# Modular latent operation (MLO): the random-target experiment

We edit the latent $z = \varphi(x)$ of a frozen text encoder through an operator on structured states, $\omega_Z(z) = \rho(z;\ s,\ \omega_S(s))$ with $s = \psi(z)$:

| piece | symbol | here |
|---|---|---|
| encoder | $\varphi$ | `Encoder("embeddinggemma", 256)`: the 256-wide prefix of the cached talent-motivation encodings |
| reader | $\psi(s \mid z)$ | `Reader(columns, hidden=2)`, cross-fitted over 5 folds with `cross_fit` |
| operator | $\omega_S$ | any function `states DataFrame -> target states DataFrame`, here `LinearSCM(...).do(X=k)` per unit |
| renderer | $\rho$ | a `Renderer` subclass: writes $s'$ back into $z$ |

**Design**

- **States.** $\psi$ reads $S^+ = \{X, T, D, U, M\}$ once. $\rho$ is fitted per state version in `STATES`; on the main one, $S = \{X, T, D, U\}$, motivation `M` stays in the residual and is held constant in $Z$-space. Its preservation is read by the independent judge $\psi_{\mathrm{eval}}$.
- **No operator in training.** $\rho$ sees observed $(z, s)$ pairs only. The minimal edit (`EditorRenderer`) learns from random targets $s' \sim \mathrm{Uniform}(S)$, so it can map any $z$ to any $s'$ and operators are swapped freely at test time.
- **Estimation gap.** Every $\rho$ is trained once on the true $s$ and once on the out-of-fold reading $\psi(z)$, and tested with both: $\psi$ = 5 folds × 5 seeds = 25 fits, $\rho$ = 6 kinds × 5 seeds × (2 arms on the main state version + 1 arm on each other one).
- **Wrong DAGs** are a test-time add-on (Section 7): same $\psi$ and $\rho$, another $\omega_S$.

The data name the sensitive attribute `X` (country-of-origin region; `O` in the paper). Fits and the evaluation are **cached** in `models/latent_lab/` (`latent_lab.cache.cached`): a cell reloads its result unless latent_lab's code, the cell's code or an input it reads changed.

Target latents $\varphi(x')$, the encodings of real texts with the target state, are used for evaluation only.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from exp.sim_talent_motivation.talent_motivation import outcome_bayes_predictions, talent_posterior_accuracy
from latent_lab import (
    ConstantRenderer, EditorRenderer, Encoder, FlowRenderer, GroupOTRenderer, IdentityRenderer, LinearSCM,
    OTEditorRenderer, OTFlowRenderer, Pipeline, Reader, StateEffectRenderer, cross_fit, evaluate, factual_score, mean_predict,
    mean_proba, state_metrics,
)
from latent_lab.cache import cached

ROOT = Path.cwd()  # the notebook sits in the repo root
SEEDS = range(5)

## 1. Data

The talent-motivation experiment (`exp/sim_talent_motivation/`): the talent SFM plus motivation `M`, a root and parent of `Y` only. Each evaluation unit has one target text, rendered with its state under its own $do(X = k)$ (`interventions.csv`).

`partition.csv` fixes the split of the 4,000 units:

| set | units | used for |
|---|---|---|
| **pool** | 2,800 in 5 folds of 560 | training $\psi$, $\rho$ and the judge |
| dev | 200 | unused here |
| **eval** | 1,000 | evaluation only; the only units with a target text $x'$ |

Within the pool, every fold holds 200 units of `L`, 200 of `P` and 160 of neither:

- **`L`** (1,000 units): the only true states $\psi$ sees. The $\psi$ of fold $k$ is fitted on `L` ∩ the other four folds (800 units) and reads all 560 units of fold $k$, so every pool unit gets an out-of-fold reading $\psi(z)$.
- **`P`** (1,000 units, disjoint from `L`): the only true states the judge $\psi_{\mathrm{eval}}$ sees. `P` sits in the folds only so that it, too, gets an out-of-fold reading.
- **$\rho$** trains on all 2,800 pool latents, with either their true states or their out-of-fold readings $\psi(z)$. So $\rho$ and the judge share the latents of `P`, and in the true-$s$ arm also their true states. $\psi$ never sees them.

Note: use `df["T"]`, never `df.T` (that is pandas' transpose).

In [3]:
DATA = ROOT / "data/sim_talent_motivation"
factual = pd.read_csv(DATA / "sim_data_factual.csv", index_col="id")
counterfactual = pd.read_csv(DATA / "sim_data_counterfactual.csv", index_col="id")  # target states, under each unit's do(X = k)
do_value = pd.read_csv(DATA / "interventions.csv", index_col="id")["do_value"]
partition = pd.read_csv(DATA / "partition.csv", index_col="id")

LEVELS = {"X": 4, "T": 3, "D": 3, "U": 3, "M": 3}
FULL = list(LEVELS)            # what psi reads

STATES = {                     # what rho conditions on
    "S1": ["X", "T", "D", "U"],
    "S2": ["X", "D", "U"],
    "S3": ["X", "T", "D", "U", "M"],
    "S4": ["X"]
}
MAIN = "S1"                    # searched, trained on both true s and psi(z); the others on psi(z) only
STATE = STATES[MAIN]           # also the operator's SCM, the joint metrics and the search

pool = partition.index[partition["partition"] == "pool"]
eval_ids = partition.index[partition["partition"] == "eval"]
folds = partition.loc[pool, "fold"].to_numpy()
labelled = partition.loc[pool, "L_rank"].gt(0).to_numpy()   # L
P = partition.index[partition["in_P"]]
assert not partition.loc[P, "L_rank"].gt(0).any(), "L and P must be disjoint"
pd.crosstab(partition.loc[pool, "fold"], partition.loc[pool, "L_rank"].gt(0).rename("in L"))

in L,False,True
fold,,
0,360,200
1,360,200
2,360,200
3,360,200
4,360,200


## 2. Encoder φ

`latents(experiment)` loads the cached encodings (no model download): `latents.factual(ids)` gives the source latents $\varphi(x)$, `latents.counterfactual(ids)` the target latents $\varphi(x')$ of the eval units.

On previous trials embeddinggemma with 768 dimensions had the best $\psi$-outputs, which is why we chose it. Smaller latent spaces and other models (Qwen and Nomic) were tried, too.

Here we use its 256-wide Matryoshka prefix (the first 256 coordinates of the cached 768-wide encodings, renormalized): at $d = 768$ the state flow's likelihood barely depends on $s$, so it hardly moves $z$. Set the width back to 768 to compare.

In [4]:
encoder = Encoder("embeddinggemma", 256)
latents = encoder.latents("talent_motivation")
z_pool, s_pool = latents.factual(pool), factual.loc[pool]
z_eval, s_eval = latents.factual(eval_ids), factual.loc[eval_ids]
z_cf, s_cf = latents.counterfactual(eval_ids), counterfactual.loc[eval_ids]
z_pool.shape, z_eval.shape

(torch.Size([2800, 256]), torch.Size([1000, 256]))

## 3. Reader ψ, cross-fitted

`cross_fit` fits one reader per fold on the labelled units (`L`) of the other four folds and reads the held-out fold, so every pool unit gets an **out-of-fold** reading $\psi(z)$ to train $\rho$ on. The eval units are read by the average of the five fold readers (`mean_predict`). Five seeds: 25 fits.

The judge $\psi_{\mathrm{eval}}$ has the same architecture, is fitted on `P`, and is used only to evaluate.

*Takes approx. 2 minutes*

In [5]:
# seed -> (out-of-fold pool readings, fold readers)
psi = {seed: cached(f"psi-seed{seed}", lambda: cross_fit(Reader(FULL, hidden=2, levels=LEVELS),
                                                       z_pool, s_pool, folds, labelled, seed=seed))
       for seed in tqdm(SEEDS, desc="psi (5 folds per seed)")}
s_read = {seed: mean_predict(readers, z_eval) for seed, (_, readers) in psi.items()}   # psi(z) on eval
judge = cached("judge", lambda: Reader(FULL, hidden=2, levels=LEVELS).fit(latents.factual(P), factual.loc[P], seed=0))


# psi on eval: per-variable accuracy, NLL and ECE; joint accuracy and NLL over S
psi_metrics = pd.concat(
    [state_metrics(mean_proba(readers, z_eval), s_eval, joint=STATE) for _, readers in psi.values()], axis=1
)
CARD = {c: int(factual[c].max()) + 1 for c in factual}
print(f"Bayes ceiling of acc_T given X, D, U and the proxies: "
      f"{talent_posterior_accuracy(s_eval, ['X', 'D', 'U', 'P', 'L', 'H', 'A'], CARD):.3f}")
oof = [(r[STATE].to_numpy() == s_pool[STATE].to_numpy()).all(1).mean() for r, _ in psi.values()]
print(f"joint accuracy over S of the out-of-fold pool readings rho trains on: {np.mean(oof):.3f}")
psi_table = pd.DataFrame({
    "psi (mean)": psi_metrics.mean(axis=1),
    "psi (sd over seeds)": psi_metrics.std(axis=1),
    "judge": judge.metrics(z_eval, s_eval, joint=STATE),
})
psi_table.round(3)

psi (5 folds per seed): 100%|██████████| 5/5 [01:33<00:00, 18.71s/it]


Bayes ceiling of acc_T given X, D, U and the proxies: 0.936
joint accuracy over S of the out-of-fold pool readings rho trains on: 0.655


,psi (mean),psi (sd over seeds),judge
acc_X,0.997,0.001,0.994
nll_X,0.019,0.001,0.022
ece_X,0.013,0.001,0.010
acc_T,0.774,0.008,0.756
nll_T,0.528,0.009,0.602
ece_T,0.045,0.006,0.084
acc_D,0.993,0.001,0.995
nll_D,0.032,0.004,0.023
ece_D,0.016,0.001,0.010
acc_U,0.892,0.012,0.910


### Robustness of ψ to the wording

Does $\psi$ read the meaning of a level or the codebook's phrases? 500 eval units were also rendered with a held-out paraphrase of the codebook ($\mathcal V_B$; `z_factual_vb.pt`) next to the training wording ($\mathcal V_A$). Per variable: accuracy on the same units in both wordings and the gap $\mathcal V_A - \mathcal V_B$ with a 95 % paired bootstrap CI over units ($\psi$: averaged over seeds). `X` keeps its wording and serves as negative control. The judge is fitted on $\mathcal V_A$ too, so it is reported alongside: a large gap for it means $\psi_{\mathrm{eval}}$ needs $\mathcal V_B$ texts as well.

In [6]:
vb = encoder.auxiliary("talent_motivation", "z_factual_vb.pt")
z_a, z_b, s_b = latents.factual(vb.ids), vb.factual(vb.ids), factual.loc[vb.ids]
boot = np.random.default_rng(0).integers(0, len(vb.ids), (2000, len(vb.ids)))   # paired: same units


def robustness(reads_a: list[pd.DataFrame], reads_b: list[pd.DataFrame]) -> pd.DataFrame:
    rows = {}
    for c in FULL:
        a = np.mean([r[c].to_numpy() == s_b[c].to_numpy() for r in reads_a], axis=0)   # per unit, over seeds
        b = np.mean([r[c].to_numpy() == s_b[c].to_numpy() for r in reads_b], axis=0)
        gaps = a[boot].mean(1) - b[boot].mean(1)
        rows[c] = {"V_A": a.mean(), "V_B": b.mean(), "gap": a.mean() - b.mean(),
                   "CI low": np.quantile(gaps, 0.025), "CI high": np.quantile(gaps, 0.975)}
    return pd.DataFrame(rows).T


robustness_table = pd.concat({
    "psi": robustness([mean_predict(r, z_a) for _, r in psi.values()], [mean_predict(r, z_b) for _, r in psi.values()]),
    "judge": robustness([judge.predict(z_a)], [judge.predict(z_b)]),
}, axis=1)
robustness_table.round(3)

psi                               judge                             
     V_A    V_B    gap CI low CI high    V_A    V_B    gap CI low CI high
X  0.995  0.992  0.003 -0.007   0.013  0.992  0.992  0.000 -0.010   0.012
T  0.788  0.739  0.049  0.015   0.083  0.760  0.714  0.046  0.008   0.084
D  0.993  0.984  0.010 -0.000   0.020  0.994  0.988  0.006 -0.004   0.016
U  0.908  0.821  0.087  0.058   0.118  0.910  0.818  0.092  0.054   0.130
M  0.545  0.517  0.028 -0.001   0.056  0.634  0.566  0.068  0.020   0.116

## 4. Renderers ρ: 6 kinds × the state versions in `STATES` × 5 seeds, searched first

- `flow`: constant-noise edit, $u = F^{-1}(z; s)$, $z' = F(u; s')$.
- `editor`: minimal edit, trained on random targets $s' \sim \mathrm{Uniform}(S)$ against a critic fitted on the same training states.
- `ot_editor`: the same minimal edit with an input-convex gradient map, $x' = \nabla_x \phi(x; s')$.
- `ot_flow`: shared-reference OT flow, $u = F_s^{-1}(z)$, $z' = F_{s'}(u)$, fitted by OT flow matching within each state.
- `state_effect`, `group_ot`: closed-form baselines (additive state effect, Gaussian OT between the groups of `X`).

**State versions.** Each $\rho$ is fitted once per version in `STATES`, i.e. per set of columns it conditions on. $\psi$ and the operator stay the same: the operator acts on the whole reading, and $\rho$ only sees its columns of $s$ and $s'$. A column left out (e.g. `T` in S2, `D` and `U` in S4) stays in the residual; one added (`M` in S3) is held at $\psi$'s reading instead of in the residual. The main version `MAIN` is searched and trained on both true $s$ and $\psi(z)$; the others reuse its selected settings and are trained on $\psi(z)$ only, as in deployment.

`FlowRenderer` and `EditorRenderer` take under a minute each on CPU at $d = 256$; the whole loop runs for a while.

### Grid search, without target latents

Each configuration is fitted once (seed 0, true states) on pool folds 1–4 and scored on fold 0 with `factual_score`: every fold-0 unit is edited to the observed state of another, randomly drawn unit.

- `fidelity`: the share of units whose whole target state $\psi$'s fold-0 reader (fitted on folds 1–4, never on these units) reads in $z'$. Not the judge, which stays reserved for the evaluation.
- `realism`: for every column and level, the energy distance between edited latents with that target level and observed latents with that level, averaged. Lower is better.

Selection per kind: among configurations within 0.05 of the best fidelity, the lowest realism. Grids: learning rate × width for the two flows, proximity for the editor, proximity × learning rate for the OT editor (20 trials). The closed-form renderers are not searched. Target latents $\varphi(x')$ are not used, so the evaluation stays untouched. Every trial is cached.

**Blind spot.** Both criteria judge the edited latents as a *population*, never the individual unit. `realism` asks whether the edited latents with target level $k$ look like real latents with level $k$, not whether unit $i$ became *its own* version with level $k$. An edit that throws away what is specific to the unit (its persona, template and wording, and everything outside the conditioning set, such as `M` and the talent proxies) and returns a typical latent of the target state therefore scores well on both. In the extreme, replacing $z$ by a random real latent with state $s'$ scores perfectly. The search cannot tell such an edit from a good one. Only the evaluation can: `gap_closed` and `distance` compare each $z'$ with the same unit's target latent $\varphi(x')$, and the judge's `acc_M` and `acc_T` check what should have stayed unchanged. Check the selected configurations there. `shift` is reported but not used for selection, because a correct edit may need to move far.

In [7]:
search_fit, search_val = folds != 0, folds == 0
z_sf, s_sf = z_pool[torch.as_tensor(search_fit)], s_pool[search_fit]
z_sv, s_sv = z_pool[torch.as_tensor(search_val)], s_pool[search_val]
selector = psi[0][1][0]   # psi's fold-0 reader: fitted on folds 1-4 only

GRID = {
    "flow": (FlowRenderer, [dict(epochs=500, lr=lr, hidden=h) for lr in (3e-4, 1e-3) for h in (68, 136)]),
    "ot_flow": (OTFlowRenderer, [dict(epochs=500, lr=lr, hidden=h) for lr in (3e-4, 1e-3) for h in (256, 512)]),
    "editor": (EditorRenderer, [dict(epochs=500, proximity=p) for p in (0.1, 0.3, 1.0, 3.0)]),
    # it learns slowly from its near-identity start, so its learning rate is searched too
    "ot_editor": (OTEditorRenderer, [dict(epochs=500, lr=lr, proximity=p) for lr in (1e-3, 1e-2) for p in (0.1, 0.3, 1.0, 3.0)]),
}
trials = [(kind, cls, config) for kind, (cls, configs) in GRID.items() for config in configs]
rows, search_fits = [], {}
for kind, cls, config in (bar := tqdm(trials, desc="search")):
    bar.set_postfix(kind=kind, **config)
    name = f"search-{kind}-" + "-".join(f"{k}{v}" for k, v in config.items())
    trial = cached(name, lambda: cls(STATE, levels=LEVELS, **config).fit(z_sf, s_sf, seed=0))
    search_fits[(kind, str(config))] = trial
    rows.append({"kind": kind, "config": config, **factual_score(trial, z_sv, s_sv, selector)})
search = pd.DataFrame(rows)


def select(trials: pd.DataFrame) -> dict:
    eligible = trials[trials["fidelity"] >= trials["fidelity"].max() - 0.05]
    return eligible.loc[eligible["realism"].idxmin(), "config"]


best = {kind: select(trials) for kind, trials in search.groupby("kind")}
print("selected:", best)
search.assign(config=search["config"].map(str)).set_index(["kind", "config"]).round(3)

search: 100%|██████████| 20/20 [12:21<00:00, 37.05s/it, epochs=500, kind=ot_editor, lr=0.01, proximity=3]   

selected: {'editor': {'epochs': 500, 'proximity': 1.0}, 'flow': {'epochs': 500, 'lr': 0.0003, 'hidden': 68}, 'ot_editor': {'epochs': 500, 'lr': 0.01, 'proximity': 0.3}, 'ot_flow': {'epochs': 500, 'lr': 0.0003, 'hidden': 512}}


fidelity  realism  \
kind      config                                                              
flow      {'epochs': 500, 'lr': 0.0003, 'hidden': 68}        0.486    0.015   
          {'epochs': 500, 'lr': 0.0003, 'hidden': 136}       0.488    0.016   
          {'epochs': 500, 'lr': 0.001, 'hidden': 68}         0.479    0.015   
          {'epochs': 500, 'lr': 0.001, 'hidden': 136}        0.486    0.015   
ot_flow   {'epochs': 500, 'lr': 0.0003, 'hidden': 256}       0.602    0.017   
          {'epochs': 500, 'lr': 0.0003, 'hidden': 512}       0.618    0.017   
          {'epochs': 500, 'lr': 0.001, 'hidden': 256}        0.545    0.017   
          {'epochs': 500, 'lr': 0.001, 'hidden': 512}        0.573    0.018   
editor    {'epochs': 500, 'proximity': 0.1}                  0.959    0.035   
          {'epochs': 500, 'proximity': 0.3}                  0.946    0.028   
          {'epochs': 500, 'proximity': 1.0}                  0.921    0.024   
          {'epochs': 500, 'proximity': 3.0}                  0.866    0.023   
ot_editor {'epochs': 500, 'lr': 0.001, 'proximity': 0.1}     0.902    0.132   
          {'epochs': 500, 'lr': 0.001, 'proximity': 0.3}     0.904    0.078   
          {'epochs': 500, 'lr': 0.001, 'proximity': 1.0}     0.886    0.047   
          {'epochs': 500, 'lr': 0.001, 'proximity': 3.0}     0.784    0.032   
          {'epochs': 500, 'lr': 0.01, 'proximity': 0.1}      0.939    0.038   
          {'epochs': 500, 'lr': 0.01, 'proximity': 0.3}      0.930    0.030   
          {'epochs': 500, 'lr': 0.01, 'proximity': 1.0}      0.884    0.026   
          {'epochs': 500, 'lr': 0.01, 'proximity': 3.0}      0.802    0.024   

                                                          shift  
kind      config                                                 
flow      {'epochs': 500, 'lr': 0.0003, 'hidden': 68}     0.404  
          {'epochs': 500, 'lr': 0.0003, 'hidden': 136}    0.385  
          {'epochs': 500, 'lr': 0.001, 'hidden': 68}      0.408  
          {'epochs': 500, 'lr': 0.001, 'hidden': 136}     0.399  
ot_flow   {'epochs': 500, 'lr': 0.0003, 'hidden': 256}    0.492  
          {'epochs': 500, 'lr': 0.0003, 'hidden': 512}    0.538  
          {'epochs': 500, 'lr': 0.001, 'hidden': 256}     0.539  
          {'epochs': 500, 'lr': 0.001, 'hidden': 512}     0.569  
editor    {'epochs': 500, 'proximity': 0.1}               0.495  
          {'epochs': 500, 'proximity': 0.3}               0.433  
          {'epochs': 500, 'proximity': 1.0}               0.346  
          {'epochs': 500, 'proximity': 3.0}               0.303  
ot_editor {'epochs': 500, 'lr': 0.001, 'proximity': 0.1}  0.797  
          {'epochs': 500, 'lr': 0.001, 'proximity': 0.3}  0.639  
          {'epochs': 500, 'lr': 0.001, 'proximity': 1.0}  0.518  
          {'epochs': 500, 'lr': 0.001, 'proximity': 3.0}  0.424  
          {'epochs': 500, 'lr': 0.01, 'proximity': 0.1}   0.531  
          {'epochs': 500, 'lr': 0.01, 'proximity': 0.3}   0.457  
          {'epochs': 500, 'lr': 0.01, 'proximity': 1.0}   0.406  
          {'epochs': 500, 'lr': 0.01, 'proximity': 3.0}   0.352

In [8]:
# searched kinds use the selected configuration; the default argument keeps each one's
# cache key to its own settings
RHOS = {
    "flow": lambda columns, c=best["flow"]: FlowRenderer(columns, levels=LEVELS, **c),
    "editor": lambda columns, c=best["editor"]: EditorRenderer(columns, levels=LEVELS, **c),
    "ot_editor": lambda columns, c=best["ot_editor"]: OTEditorRenderer(columns, levels=LEVELS, **c),
    "ot_flow": lambda columns, c=best["ot_flow"]: OTFlowRenderer(columns, levels=LEVELS, **c),
    "state_effect": lambda columns: StateEffectRenderer(columns, levels=LEVELS),
    "group_ot": lambda columns: GroupOTRenderer(columns, on="X", shrink=0.1, levels=LEVELS),
}
runs = [(S, kind, arm, seed) for S in STATES for seed in SEEDS
        for arm in (("true s", "ψ(z)") if S == MAIN else ("ψ(z)",)) for kind in RHOS]
rho = {}   # (state version, kind, train states, seed) -> fitted renderer
for S, kind, arm, seed in (bar := tqdm(runs, desc="rho")):
    bar.set_postfix(S=S, kind=kind, train=arm, seed=seed)
    states, columns = (s_pool if arm == "true s" else psi[seed][0]), STATES[S]
    make = RHOS[kind]  # only this renderer's settings enter the cache key, not all of RHOS
    name = f"rho-{S}-{kind}-{'true' if arm == 'true s' else 'psi'}-seed{seed}"
    rho[(S, kind, arm, seed)] = cached(name, lambda: make(columns).fit(z_pool, states, seed=seed))
len(rho)

rho: 100%|██████████| 150/150 [1:15:57<00:00, 30.39s/it, S=S4, kind=group_ot, seed=4, train=ψ(z)]     


150

## 5. Operator $\omega_S$

$\rho$ was fitted without any operator; $\omega_S$ enters only now, between reading and rendering. Each eval unit's $do(X = k)$ on the SCM over $S$, with the numbers copied from `exp/sim_talent/talent_sfm.py` (**editing them changes only the operator, not the data**). Columns outside the SCM, such as `M`, pass through unchanged. `operator(seed)` builds a freshly seeded SCM, so every run sees the same abduction draws.

The SCM operator follows abduction (noise drawn from the range consistent with the observed state), action (set the do() values) and prediction (recompute the downstream nodes); `blocked={"D": ["X"]}` makes `D` read the observed `X`.

In [9]:
PARENTS = {"D": {"X": 0.30, "T": 0.60}, "U": {"X": 0.35, "T": 0.50}}   # node: {parent: weight}
NOISE = {"D": (-0.10, 0.50), "U": (-0.15, 0.50)}                        # node: (mu, sd)
do_eval = do_value.loc[eval_ids].to_numpy()


def operator(seed=0, parents=PARENTS, blocked=None):
    scm = LinearSCM({c: LEVELS[c] for c in STATE}, parents, NOISE, seed=seed)
    return scm.do(X=lambda s: do_eval, blocked=blocked)


print(LinearSCM({c: LEVELS[c] for c in STATE}, PARENTS, NOISE))
# sanity check: the operator on the true source states vs. the simulated target states
pd.DataFrame({
    "operator": operator()(s_eval[STATE].reset_index(drop=True))["U"].value_counts(normalize=True),
    "simulated": s_cf["U"].value_counts(normalize=True),
}).sort_index().round(3)

LinearSCM, roots ['X', 'T']
  D = clip(round(0.30·X + 0.60·T + ε), 0, 2),  ε ~ N(-0.10, 0.50)
  U = clip(round(0.35·X + 0.50·T + ε), 0, 2),  ε ~ N(-0.15, 0.50)


,operator,simulated
U,,
0,0.338,0.317
1,0.479,0.507
2,0.183,0.176


## 6. Downstream predictor π

A frozen MLP $\pi: Z \to Y$ (a one-column `Reader`), fitted on the observed pool latents and their outcome `Y`, with the epoch selected on a held-out 20 % of the pool. It never sees target data. References: $\pi$ on the real $\varphi(x')$, and the Bayes-optimal accuracy for $Y'$ given the true counterfactual state with and without `M`.

`score(z')` collects every metric of one edit, against the true target.

In [10]:
pi = cached("pi", lambda: Reader(["Y"], hidden=2, levels={"Y": CARD["Y"]}).fit(z_pool, s_pool, seed=0))


def predict(z: torch.Tensor) -> torch.Tensor:
    return pi.proba(z)["Y"].argmax(-1)


def score(z_prime: torch.Tensor) -> pd.Series:
    return evaluate(z_eval, z_prime, z_cf, judge, s_cf, joint=STATE,
                    predict=predict, outcome=s_cf["Y"], groups=s_cf["X"])


y_cf = s_cf["Y"].to_numpy()
outcome_table = pd.Series({
    "pi(z) vs factual Y": float((predict(z_eval).numpy() == s_eval["Y"].to_numpy()).mean()),
    "pi(phi(x')) vs Y'": float((predict(z_cf).numpy() == y_cf).mean()),
    **{f"Bayes vs Y' | {name}": float((p.argmax(1) == y_cf).mean())
       for name, p in outcome_bayes_predictions(s_cf).items()},
})
outcome_table.round(3)

pi(z) vs factual Y                                             0.665
pi(phi(x')) vs Y'                                              0.639
Bayes vs Y' | all parents (T, M known)                         0.783
Bayes vs Y' | state only (T known, M unknown)                  0.680
Bayes vs Y' | text content (X, D, U, M, proxies)               0.773
Bayes vs Y' | text content without M                           0.663
Bayes vs Y' | no talent or motivation information (X, D, U)    0.652
dtype: float64

## 7. Evaluation under each unit's do(X = k)

Every $\rho$ is tested with the true source states and with $\psi(z)$, per state version `S`. Train × test gives the estimation gap: the `true s / true s` cell is the privileged upper bound, `ψ(z) / ψ(z)` is deployment.

- **Fidelity and preservation:** the judge's `joint_acc` over $S$ and `acc_*` on $z'$ against the true target states; `acc_M` and `acc_T` are preservation (neither changes under $do(X)$, and $\rho$ never sees `M`). `real_*` is the judge on $\varphi(x')$ itself: reading $z'$ *better* than a real latent points to off-manifold edits.
- **Latent:** `gap_closed` to the paired $\varphi(x')$ (1 = perfect, 0 = no better than $z$), `distance` $|z' - \varphi(x')|$ (the energy score of a point edit), `energy` distance between the sets $\{z'\}$ and $\{\varphi(x')\}$, `shift` $|z' - z|$; all in standardized coordinates, divided by $\sqrt d$.
- **Downstream:** `down_acc` of $\pi(z')$ against $Y'$, `down_agree` with $\pi(z)$, and `parity_gap`, the range over the target regions of the mean predicted $Y$.

In [11]:
def evaluate_all() -> pd.DataFrame:
    rows = []
    for (S, kind, arm, seed), renderer in tqdm(rho.items(), desc="evaluate", total=len(rho)):
        for test, states in {"true s": s_eval, "ψ(z)": s_read[seed]}.items():
            result = Pipeline(None, operator(seed), renderer)(z_eval, states)
            rows.append({"S": S, "rho": kind, "train": arm, "test": test, "seed": seed, **score(result.z_prime)})
    for kind, renderer in {"identity": IdentityRenderer(STATE),
                           "constant": ConstantRenderer(STATE, on="X").fit(z_pool, s_pool)}.items():
        result = Pipeline(None, operator(), renderer)(z_eval, s_eval)
        rows.append({"S": MAIN, "rho": kind, "train": "-", "test": "true s", "seed": 0, **score(result.z_prime)})
    return pd.DataFrame(rows)


results = cached("results", evaluate_all)

metrics = ["joint_acc", "acc_X", "acc_D", "acc_U", "acc_T", "acc_M",
           "gap_closed", "distance", "energy", "shift", "down_acc", "down_agree", "parity_gap"]
print("references on the real phi(x'):", results.filter(like="real_").iloc[0].round(3).to_dict())
summary = results.groupby(["S", "rho", "train", "test"])[metrics].agg(["mean", "std"])
summary.xs("mean", axis=1, level=1).round(3)

evaluate: 100%|██████████| 150/150 [02:02<00:00,  1.22it/s]

references on the real phi(x'): {'real_acc_X': 0.991, 'real_acc_T': 0.76, 'real_acc_D': 0.989, 'real_acc_U': 0.923, 'real_acc_M': 0.61, 'real_joint_acc': 0.686, 'real_down_acc': 0.639, 'real_down_agree': 0.548, 'real_parity_gap': 0.773}


joint_acc  acc_X  acc_D  acc_U  acc_T  acc_M  \
S  rho          train  test                                                   
S1 constant     -      true s      0.125  0.500  0.560  0.502  0.762  0.624   
   editor       true s true s      0.471  0.997  0.666  0.696  1.000  0.595   
                       ψ(z)        0.347  0.997  0.653  0.652  0.774  0.603   
                ψ(z)   true s      0.452  0.988  0.665  0.694  0.973  0.577   
                       ψ(z)        0.343  0.987  0.652  0.650  0.771  0.585   
...                                  ...    ...    ...    ...    ...    ...   
S4 ot_editor    ψ(z)   ψ(z)        0.238  0.990  0.561  0.525  0.709  0.606   
   ot_flow      ψ(z)   true s      0.259  0.994  0.576  0.533  0.746  0.623   
                       ψ(z)        0.260  0.993  0.576  0.534  0.746  0.623   
   state_effect ψ(z)   true s      0.257  1.000  0.556  0.532  0.751  0.629   
                       ψ(z)        0.258  0.999  0.556  0.533  0.751  0.629   

                               gap_closed  distance  energy  shift  down_acc  \
S  rho          train  test                                                    
S1 constant     -      true s       0.099     0.712   0.002  0.266     0.551   
   editor       true s true s       0.099     0.713   0.002  0.364     0.631   
                       ψ(z)         0.098     0.714   0.003  0.364     0.592   
                ψ(z)   true s       0.085     0.723   0.003  0.389     0.627   
                       ψ(z)         0.086     0.723   0.003  0.385     0.588   
...                                   ...       ...     ...    ...       ...   
S4 ot_editor    ψ(z)   ψ(z)         0.126     0.694   0.002  0.359     0.562   
   ot_flow      ψ(z)   true s       0.196     0.637   0.003  0.578     0.591   
                       ψ(z)         0.196     0.637   0.003  0.578     0.591   
   state_effect ψ(z)   true s       0.165     0.660   0.002  0.432     0.617   
                       ψ(z)         0.165     0.660   0.002  0.431     0.618   

                               down_agree  parity_gap  
S  rho          train  test                            
S1 constant     -      true s       0.760       0.525  
   editor       true s true s       0.621       0.555  
                       ψ(z)         0.638       0.508  
                ψ(z)   true s       0.638       0.470  
                       ψ(z)         0.671       0.412  
...                                   ...         ...  
S4 ot_editor    ψ(z)   ψ(z)         0.765       0.430  
   ot_flow      ψ(z)   true s       0.636       0.607  
                       ψ(z)         0.636       0.606  
   state_effect ψ(z)   true s       0.586       0.757  
                       ψ(z)         0.586       0.753  

[62 rows x 13 columns]

In [12]:
# estimation gap per renderer (main state version): train states x test states
main = results[(results["S"] == MAIN) & (results["train"] != "-")]
main.pivot_table(["joint_acc", "gap_closed", "down_acc"], "rho", ["train", "test"]).round(3)

down_acc                      gap_closed                       \
train          true s          ψ(z)            true s          ψ(z)          
test           true s   ψ(z) true s   ψ(z)     true s   ψ(z) true s   ψ(z)   
rho                                                                          
editor          0.631  0.592  0.627  0.588      0.099  0.098  0.085  0.086   
flow            0.609  0.605  0.613  0.611      0.172  0.171  0.173  0.172   
group_ot        0.586  0.587  0.588  0.589      0.179  0.179  0.178  0.178   
ot_editor       0.616  0.582  0.612  0.574      0.102  0.102  0.093  0.094   
ot_flow         0.615  0.612  0.613  0.607      0.192  0.190  0.180  0.179   
state_effect    0.615  0.612  0.616  0.612      0.166  0.165  0.164  0.164   

             joint_acc                       
train           true s          ψ(z)         
test            true s   ψ(z) true s   ψ(z)  
rho                                          
editor           0.471  0.347  0.452  0.343  
flow             0.344  0.325  0.331  0.315  
group_ot         0.259  0.260  0.258  0.259  
ot_editor        0.471  0.347  0.465  0.345  
ot_flow          0.344  0.328  0.344  0.328  
state_effect     0.358  0.339  0.350  0.332

In [13]:
# state versions side by side: trained and tested on psi(z), mean over seeds
deploy = results[(results["train"] == "ψ(z)") & (results["test"] == "ψ(z)")]
deploy.pivot_table(["gap_closed", "acc_X", "acc_D", "acc_U", "acc_T", "acc_M"], "rho", "S").round(3)

acc_D                       acc_M                       acc_T  \
S                S1     S2     S3     S4     S1     S2     S3     S4     S1   
rho                                                                           
editor        0.652  0.652  0.651  0.564  0.585  0.606  0.582  0.620  0.771   
flow          0.653  0.654  0.654  0.558  0.627  0.625  0.625  0.627  0.721   
group_ot      0.557  0.557  0.557  0.557  0.609  0.609  0.609  0.609  0.731   
ot_editor     0.654  0.653  0.652  0.561  0.521  0.550  0.565  0.606  0.775   
ot_flow       0.653  0.654  0.653  0.576  0.626  0.625  0.625  0.623  0.735   
state_effect  0.654  0.654  0.654  0.556  0.630  0.630  0.630  0.629  0.735   

                     ...  acc_U         acc_X                      gap_closed  \
S                S2  ...     S3     S4     S1     S2     S3     S4         S1   
rho                  ...                                                        
editor        0.622  ...  0.649  0.527  0.987  0.975  0.989  0.973      0.086   
flow          0.696  ...  0.629  0.537  0.996  0.993  0.985  0.997      0.172   
group_ot      0.731  ...  0.553  0.553  0.998  0.998  0.998  0.998      0.178   
ot_editor     0.596  ...  0.651  0.525  0.995  0.985  0.996  0.990      0.094   
ot_flow       0.715  ...  0.640  0.534  0.994  0.993  0.990  0.993      0.179   
state_effect  0.696  ...  0.655  0.533  0.997  0.998  0.997  0.999      0.164   

                                   
S                S2     S3     S4  
rho                                
editor        0.104  0.069  0.122  
flow          0.171  0.171  0.178  
group_ot      0.178  0.178  0.178  
ot_editor     0.107  0.066  0.126  
ot_flow       0.175  0.182  0.196  
state_effect  0.163  0.163  0.165  

[6 rows x 24 columns]

## 8. Add-on: wrong DAGs at test time

$\psi$ and $\rho$ stay fixed; only $\omega_S$ changes. Shown for the flow on the main state version, trained and tested on $\psi(z)$, seed 0, against the true target.

In [14]:
wrong = {
    "true SCM": operator(),
    "direct effect (D, U blocked)": operator(blocked={"D": ["X"], "U": ["X"]}),
    "wrong SCM (no X->D)": operator(parents={**PARENTS, "D": {"T": 0.60}}),
}
renderer = rho[(MAIN, "flow", "ψ(z)", 0)]
wrong_table = pd.DataFrame({
    name: score(Pipeline(None, op, renderer)(z_eval, s_read[0]).z_prime)
    for name, op in wrong.items()
}).T
wrong_table[metrics].round(3)

,joint_acc,acc_X,acc_D,acc_U,acc_T,acc_M,gap_closed,distance,energy,shift,down_acc,down_agree,parity_gap
true SCM,0.315,0.997,0.666,0.617,0.711,0.622,0.177,0.651,0.001,0.462,0.615,0.578,0.792
"direct effect (D, U blocked)",0.258,0.995,0.563,0.538,0.696,0.614,0.173,0.655,0.002,0.420,0.584,0.778,0.297
wrong SCM (no X->D),0.269,0.999,0.561,0.619,0.715,0.619,0.174,0.654,0.001,0.431,0.587,0.717,0.440


## 9. Store the run

One JSON file per run in `reports/latent_lab/`: the settings, the code version (git commit, and whether `latent_lab/` had uncommitted changes) every table above at full precision, and the training time and convergence of every fit (seconds, epochs run, the epoch kept, early stopping, and the loss per epoch). Times are those of the original fit, also for models reloaded from the cache. The fitted models stay in the cache (`models/latent_lab/`).

In [15]:
import json
import subprocess
from datetime import datetime


def table(frame) -> list | dict:
    # DataFrames as one record per row, index included; Series as a dict
    if isinstance(frame, pd.Series):
        return {str(k): v for k, v in frame.items()}
    frame = frame.copy()
    frame.columns = [" | ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in frame.columns]
    frame = frame if isinstance(frame.index, pd.RangeIndex) else frame.reset_index()
    return frame.to_dict(orient="records")


def git(*args: str) -> str:
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True).stdout.strip()


run = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "git_commit": git("rev-parse", "HEAD"),
    "latent_lab_uncommitted_changes": bool(git("status", "--porcelain", "latent_lab")),
    "settings": {
        "encoder": repr(encoder), "seeds": list(SEEDS), "states": STATES, "main": MAIN, "read": FULL, "levels": LEVELS,
        "units": {"pool": len(pool), "labelled": int(labelled.sum()), "judge": len(P), "eval": len(eval_ids)},
        "search_grid": {kind: configs for kind, (_, configs) in GRID.items()},
        "selected": best,
        "renderers": {kind: {"class": type(r).__name__, **{k: v for k, v in vars(r).items() if not k.startswith("_")}}
                      for kind, r in ((kind, make(STATE)) for kind, make in RHOS.items())},
    },
    "psi": table(psi_table),
    "robustness": table(robustness_table),
    "search": table(search.assign(config=search["config"].map(str))),
    "outcome": table(outcome_table),
    "results": table(results),
    "summary": table(results.groupby(["S", "rho", "train", "test"])[metrics].agg(["mean", "std"])),
    "wrong_dag": table(wrong_table),
}
# training time and convergence of every fit (as originally trained, also when reloaded from the cache)
fits = (
    [{"model": "psi", "seed": seed, "fold": fold, **r.fit_log} for seed, (_, readers) in psi.items() for fold, r in enumerate(readers)]
    + [{"model": "judge", **judge.fit_log}, {"model": "pi", **pi.fit_log}]
    + [{"model": f"search-{kind}", "config": config, **t.fit_log} for (kind, config), t in search_fits.items()]
    + [{"model": f"rho-{kind}", "S": S, "train": arm, "seed": seed, **r.fit_log} for (S, kind, arm, seed), r in rho.items()]
)
run["training"] = fits
path = ROOT / "reports" / "latent_lab" / f"run-{datetime.now():%Y%m%d-%H%M%S}.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(run, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("stored", path)
training_table = pd.DataFrame([{k: v for k, v in f.items() if k != "history"} for f in fits])
print(f"total training time: {training_table['seconds'].sum() / 60:.1f} min")
training_table.groupby("model")[["seconds", "epochs_run", "best_epoch"]].agg(["mean", "max"]).round(1)

stored /Users/kestel/Projects/phd/latent_intervention/reports/latent_lab/run-20261024-164358.json
total training time: 89.8 min


seconds        epochs_run        best_epoch       
                    mean    max       mean    max       mean    max
model                                                              
judge                5.8    5.8      212.0  212.0      182.0  182.0
pi                   2.3    2.3       49.0   49.0       19.0   19.0
psi                  3.7    5.3      176.1  244.0      146.1  214.0
rho-editor          21.0   28.6      445.1  500.0      428.2  500.0
rho-flow            29.3   34.1       97.6  110.0       67.6   80.0
rho-group_ot         0.0    0.0        NaN    NaN        NaN    NaN
rho-ot_editor       41.7   53.9      437.8  500.0      414.1  500.0
rho-ot_flow         90.2  101.9      492.9  500.0      482.2  500.0
rho-state_effect     0.0    0.0        NaN    NaN        NaN    NaN
search-editor       18.0   20.0      413.2  500.0      396.5  497.0
search-flow         18.5   24.3       61.8   94.0       31.8   64.0
search-ot_editor    38.2   42.3      452.5  500.0      442.2  500.0
search-ot_flow      70.7   76.0      483.8  500.0      472.0  495.0